# Algebraic numbers on a Colab GPU

1. Runtime → Change runtime type → Hardware accelerator → **T4 GPU** (free) or **A100** (paid).
2. Runtime → Run all.

The first cell clones this repo if `algebraics/` is missing. GPU work is batched **float64** companion-matrix eigenvalues. That is the highest precision CUDA gives you.

What adds detail is a larger `MAXH`. Brooks used 15. A free T4 is enough for 17. An A100 can do 18.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("algebraics/algebraics.py").exists():
    !git clone --depth 1 https://github.com/ST-48-1240162/Constellatio-Numerorum.git /content/Constellatio-Numerorum
    os.chdir("/content/Constellatio-Numerorum")

import torch
print("cwd", Path.cwd())
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), end=" ")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("(enable a GPU runtime or this falls back to CPU)")

In [ ]:
MAXH = 17
WIDTH = 3840
BATCH = 8192

from algebraics.algebraics_gpu import load_or_compute_gpu, pick_device, render_wiki

device = pick_device()
points = load_or_compute_gpu(MAXH, device=device, batch=BATCH)
print("roots", len(points))

out = Path("algebraics") / f"algebraics_h{MAXH}_{WIDTH}px.png"
render_wiki(points, out, width=WIDTH)
print("wrote", out)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

png = Path("algebraics") / f"algebraics_h{MAXH}_{WIDTH}px.png"
display(Image(filename=str(png), width=960))

if IN_COLAB:
    from google.colab import files
    files.download(str(png))